# 04 Operate traces and audit
## Learning objective
Inspect correlation, OTel spans, completeness and tamper detection without confusing application evidence with native product signals.
## Technical level
L300-L500. Expected duration: 15 minutes.
## Prerequisites and setup
Notebook 00; Python 3.12 `.venv`. Bash/PowerShell: `uv sync --frozen --all-extras --dev`. Read the [notebook guide](README.md) for lab order, consent rules and the production/WAF disclaimer.
## Cost and permissions
These cells are offline and require no Azure permissions. Optional live invocation and read-only ingestion query below are default-off, independently authorized and billable.

## Operate with an evidence ladder

Observability has several independent completion points. Creating a local
span proves that application instrumentation ran. An exporter reporting
successful submission proves something narrower than ingestion. A scoped
workspace query returning correlated rows is stronger evidence of destination
visibility, but still does not prove native Agent 365 activity, Defender
detection or Purview enforcement. Start every operational explanation by
naming the plane, scope, time window and correlation value actually observed.

Application correlation IDs and W3C trace context solve different problems.
A validated correlation ID groups application artifacts and query results.
A traceparent carries distributed tracing context; its shape alone is not
evidence that a downstream product retained the trace. Neither is an Entra
identity or Agent 365 registry identifier. You need actual native identifiers
and timestamps to reconcile native product observations later.

This lab first creates a local DRY_RUN workflow so its required stages can be
inspected without a cloud dependency. It then damages only a named copy of the
audit to demonstrate tamper detection. Do not alter the original evidence.
Finally, two independent default-off extensions allow a synthetic presentation
invocation and a read-only Monitor query. Export must already be configured and
privacy-reviewed outside the notebook. Export configuration requires separate reviewed approval;
the new invocation requires independent opt-in for already-configured egress. We do not
change destinations, retention,
permissions, account logging, consent or native publishing to produce a chart.

### Step 1 — Local completeness and an isolated tampered copy

**Exact implementation:** `workflow.run` accepts a `contracts.UserRequest`, a
run directory, explicit `config.Settings.local()`, and local
`demo_approval=True`. `workflow.REQUIRED_STAGES`,
`observability.read_stage_names` and `evaluation.trace_completeness` inspect
recorded stages. `audit.verify_audit` and `audit.audit_failure_line` verify
chained records. The returned mapping includes telemetry-export status,
correlation/trace context, artifact paths and an audit head.

Assertions prove this run remains LOCAL_ONLY, records expected context shape,
includes the specified full DRY_RUN stages, verifies the original chain and
identifies line two as the first broken record in the altered copy. They do
not prove resistance to an attacker who can rewrite the entire chain and its
stored head. Retain the head independently under appropriate access controls.
The full-write stage list also must not be blindly applied to an
AWAITING_APPROVAL run: missing writer stages there may be correct behavior.
If the original fails verification, stop and investigate before collecting
additional evidence in that directory.

In [ ]:
from pathlib import Path
from uuid import uuid4

from foundry_security_accelerator.audit import audit_failure_line, verify_audit
from foundry_security_accelerator.config import Settings
from foundry_security_accelerator.contracts import Outcome, UserRequest
from foundry_security_accelerator.evaluation import trace_completeness
from foundry_security_accelerator.observability import read_stage_names
from foundry_security_accelerator.workflow import REQUIRED_STAGES, run

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
directory = ROOT / ".fsa" / "notebooks" / str(uuid4())
result = run(
    UserRequest(text="Explain equipment policy.", subject="synthetic-analyst"),
    directory,
    settings=Settings.local(),
    demo_approval=True,
)
assert result["telemetry_export"] == "LOCAL_ONLY"
assert result["correlation_id"] and len(result["traceparent"]) == 55
assert (
    trace_completeness(REQUIRED_STAGES, read_stage_names(Path(result["traces"]))).outcome
    == Outcome.PASS
)
assert verify_audit(Path(result["audit"]), result["audit_head"])
original = Path(result["audit"]).read_text()
tampered = directory / "tampered-copy.jsonl"
tampered.write_text(original.replace("input-security", "input-securitx", 1))
assert audit_failure_line(tampered) == 2
assert verify_audit(Path(result["audit"]), result["audit_head"])
print(
    {"export": result["telemetry_export"], "complete": True, "tampered_copy_first_broken_line": 2}
)

## Optional read-only cloud observation (manual)
See [trace architecture](../docs/observability/trace-architecture.md) for private Application Insights settings and export consent. Never display connection strings. `FSA_EXPORT_MONITOR` is an Azure-mode export capability, not ingestion evidence. Configuring/enabling export requires separate reviewed approval; invocation consent alone does not grant it. With approved workspace query permissions and your own configured environment, run `uv run fsa observe --correlation-id VALUE` in a private terminal using an actual run value. Only the new independently consented default-off query cell below queries Azure; the local path never does.
## Expected results and validation
LOCAL_ONLY, complete stages for the explicitly approved DRY_RUN case, verified original audit and tampered-copy failure at line 2. A normal AWAITING_APPROVAL run intentionally lacks approval/writer stages, so a generic all-stage completeness check need not pass.
## Security explanation
A hash chain needs an independently retained head/anchor against wholesale rewriting. Local traces, export attempts, observed destination rows, Agent 365 activity, Defender alerts and Purview events are six distinct evidence scopes. Never promote one into another. Report centrally; enforce natively in each cloud.
## Troubleshooting
No rows or partial cloud query: insufficient evidence, not proof nothing happened. Export failure: preserve local evidence and investigate; do not claim delivery succeeded.
## Cleanup
Never modify original audit files. Keep only the named tampered copy for teaching, and clear outputs. [Cleanup](../docs/participant-lab/cleanup.md).
## Source modules
[observability](../src/foundry_security_accelerator/observability.py), [audit](../src/foundry_security_accelerator/audit.py), [workflow](../src/foundry_security_accelerator/workflow.py).

## Optional independent step A — Live request and application export status

**Exact import:** `presentation_live.run_question`. Inputs are dedicated pinned
settings, a fictional policy read, synthetic role and fresh local state path.
Output is an observed run mapping. Export configuration requires separate reviewed approval;
this cell requires independent default-off consent for already-configured
Monitor/OTLP egress before invoking the agent.
Invocation consent alone includes model/Search usage and local evidence. The printed field
is the application's export report, not ingestion. A later telemetry proposal
needs separate review and authorization; this prompt cannot grant it.

A missing export field remains UNREPORTED. If it says
SENT_NOT_INGESTION_PROOF, retain that wording. To query actual ingestion, take
the returned real correlation_id privately into step B, which initializes its
own settings and prompts separately. Do not substitute a historical sample,
local run ID or Agent 365 registry ID. No assertion equates a good answer with
trace completeness or native activity. False skips all live work and prompts.

In [ ]:
RUN_AZURE_LIVE = False
if RUN_AZURE_LIVE:
    from pathlib import Path
    from uuid import uuid4

    from foundry_security_accelerator.config import Settings
    from foundry_security_accelerator.presentation_live import run_question

    if input(
        "Authorize bounded synthetic invocation and local evidence only? Type RUN LIVE OPERATE: "
    ) != ("RUN LIVE OPERATE"):
        raise PermissionError("Live Operate invocation not authorized")
    operate_root = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
    operate_settings = Settings(mode="azure", _env_file=operate_root / ".env")
    AUTHORIZE_TELEMETRY_EXPORT = False  # Independent opt-in after prior review
    if operate_settings.export_monitor or operate_settings.otlp_endpoint:
        if not AUTHORIZE_TELEMETRY_EXPORT:
            raise PermissionError(
                "Configured egress requires separate reviewed approval and opt-in"
            )
        if input(
            "Confirm prior review of configured telemetry destination/content/cost. "
            "Type APPROVED TELEMETRY EGRESS: "
        ) != ("APPROVED TELEMETRY EGRESS"):
            raise PermissionError("Telemetry egress not authorized")
    if (
        not operate_settings.presentation_agent_version
        or str(operate_settings.presentation_agent_version).lower() == "latest"
    ):
        raise ValueError("Pin the dedicated presentation version before invoking")
    operate_state = operate_root / ".fsa" / "notebooks" / str(uuid4())
    observed_operate = run_question(
        operate_settings,
        text="Summarize the synthetic retention policy and cite the approved policy section.",
        role="employee",
        state_dir=operate_state,
        authorized=True,
    )
    print(
        {
            "application_status": observed_operate.get("status", "UNREPORTED"),
            "application_export": observed_operate.get("trace_export_status", "UNREPORTED"),
            "azure_monitor_ingestion": "NOT_QUERIED",
            "native_agent365": "NOT_CHECKED",
        }
    )
else:
    print("Live Operate invocation NOT_EXECUTED")

## Optional independent step B — Query actual Azure Monitor ingestion

**Exact import:** `observability.query_azure_traces`, the existing read-only
application Logs query. Inputs are private Azure settings and a real recent
application correlation ID entered after independent query authorization.
Output is the query result mapping. This separate cell does not depend on
step A's variables and can query an already authorized run.

The existing implementation bounds the workspace query to its documented
one-hour window and 50-row projection. It is not a tenant-wide search. The
notebook prints status and returned-row count, not raw rows, resource IDs or
connection strings. A missing rows field stays UNREPORTED; zero/partial rows
are insufficient evidence, not proof that no activity happened. No automatic
ingestion-delay polling or broadening of the query is authorized. A query
result still cannot establish native A365 activity, Defender or Purview.

In [ ]:
RUN_AZURE_LIVE = False
if RUN_AZURE_LIVE:
    from pathlib import Path

    from foundry_security_accelerator.config import Settings
    from foundry_security_accelerator.observability import query_azure_traces

    if input("Authorize one scoped read-only Logs query and cost? Type QUERY LIVE TRACES: ") != (
        "QUERY LIVE TRACES"
    ):
        raise PermissionError("Azure trace query not authorized")
    trace_root = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
    trace_settings = Settings(mode="azure", _env_file=trace_root / ".env")
    trace_correlation = input(
        "Enter the actual recent application correlation ID privately: "
    ).strip()
    observed_query = query_azure_traces(settings=trace_settings, correlation_id=trace_correlation)
    observed_rows = observed_query.get("rows")
    print(
        {
            "azure_monitor_query": observed_query.get("status", "UNREPORTED"),
            "returned_rows": len(observed_rows)
            if isinstance(observed_rows, (list, tuple))
            else "UNREPORTED",
            "native_agent365": "NOT_CHECKED",
            "defender_purview": "NOT_CHECKED",
        }
    )
else:
    print("Azure Monitor query NOT_EXECUTED; native Agent 365 remains separate")

## Interpretation table: keep six observation planes separate

| Plane | Observable evidence | Does not establish |
|---|---|---|
| Local application | JSONL spans/audit and retained head | Remote delivery |
| Application export | SENT_NOT_INGESTION_PROOF or explicit failure | Queryable destination rows |
| Azure Monitor ingestion | Correlated rows in the selected workspace/window | Native A365 activity |
| Native Agent 365 | Independently observed eligible activity | Defender alert or authenticated application approval |
| Defender | Actual scoped finding/detection | Every denied tool triggers an alert |
| Purview | Supported scoped audit/DLP observation | Application classification equals enterprise DLP |

Empty or partial workspace results are INSUFFICIENT_EVIDENCE, not proof nothing
happened. Query delay, scope mismatch, export failure, unsupported tables and
access restrictions are distinct possibilities. Do not select one cause without
evidence. Bounded query timing is intentionally not an automatic polling loop;
another query needs another explicit decision and may incur additional cost.

## Questions and practical exercises

Why is a valid traceparent not ingestion proof? Which evidence would detect a
complete audit rewrite if the attacker could also replace the stored head?
Compare expected stages for an awaiting-approval request and a DRY_RUN request.
Explain why configuring a Monitor connection string cannot validate Agent 365.
Draw the application exporter and native platform collection as separate
arrows; add independent authorization, retention and scope labels to each.

## Least privilege, cost, retention and cleanup

The live request needs already authorized Foundry/Search access. The query
needs read/query permission on the particular Logs workspace, not broad
subscription ownership. Approve telemetry content and egress before enabling
any exporter. Keep prompts, credentials, connection strings and customer data
out of notebook output; the query cell prints status and row count only.
An application query does not add Agent 365 support, native consent or registry
publication.

If a new native observability integration is later required, consult the
separate Microsoft OpenTelemetry Distro guidance cited below. That is a
separately reviewed implementation change, not an instruction to replace the
current application exporter or install deprecated Agent 365 packages.
Model calls, telemetry ingestion, retention and Logs queries may be billed.
Stop once evidence is sufficient, retain the original audit and independent
anchor, remove only the teaching copy when approved, restore False, and clear
outputs. No cleanup or logging configuration mutation runs in these cells.

## Live report contract and bounded execution

`presentation_live.run_question(...)` returns observed application evidence:
`status`, `model_calls`, `tool_calls`, `citations`, `quarantined_sections`,
`correlation_id`, `audit`, `audit_head`, `audit_verified`, `trace_path` and
`trace_export_status`. These field names are specific to PRESENTATION:
do not substitute the equipment API's `traces` or `telemetry_export`.
An expected denial can be a truthful safe outcome; an exception, failed
structural check or missing field must not be relabeled successful.

`run_scenes(...)` returns `status`, `scenes` and `manifest_path`.
Per-scene structural checks are not cloud model-judge scores. The runtime
is serial, bounded to three model requests and six proposed tool calls per
scene/question; the third/final round uses `tool_choice=none`.
Proposed calls can be denied before execution. Six scenes therefore do
not imply six calls or identical tool sequences, and repeated runs repeat
cost. Plan the initial verification within 18 model calls total.

All examples are opt-in. If already-configured application export is
intended, its destination, content, retention and cost require prior
independent review; then deliberately set the nested
`AUTHORIZE_TELEMETRY_EXPORT` flag and confirm its separate prompt.
No notebook configures export or changes native account logging.
A returned export status is still not ingestion. Query actual correlated
rows separately; inspect native Agent 365 activity independently.

## Read saved LIVE evidence without substituting local fixtures

In a separately authorized private terminal, the existing CLI can inspect
actual retained live evidence:

```text
uv run fsa --mode azure audit verify
uv run fsa --mode azure trace list
uv run fsa --mode azure trace show S4
uv run fsa azure agent-status --profile presentation
```

Azure-mode audit verification reads the latest LIVE session, and Azure-mode
trace list/show reads saved live scenes/spans. `S4` selects a saved scene;
if it is absent, retain the absence rather than substituting a local
fixture or fabricated trace. PRESENTATION reports store the local span
reference in `trace_path`. Saved spans are application evidence, not proof
of exporter delivery, workspace ingestion or native product activity.

For existing Agent 365 read-only commands, add `--profile presentation`
when that is the intended dedicated agent rather than EQUIPMENT, for example
`uv run fsa agent365 status --profile presentation`.
Profile selection does not authorize preview reads, native changes,
publication, consent or role assignments. Review the existing preview
prompt independently. Account enablement, registry, activity, Defender and
Purview remain separate evidence scopes; no environment result is embedded
in this notebook.

## Current platform references and claim boundaries

References checked on 2026-11-12; these are reading links, not remote code to execute.
[Foundry function calling](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)
documents the model-request/client-execution loop. The PRESENTATION profile attaches
four function definitions directly; no Toolbox is required. The model requests
`search_policies`, `get_policy_section`, `get_ticket_status`, or
`create_records_request`; application policy checks each proposed execution first.
Function selection is not authorization. Only the approved synthetic Azure Search
corpus is eligible, and a records request enters a local persistent approval queue.
Even in Azure mode, supplied action receipts are **DRY_RUN**, and the identity label
is **SYNTHETIC_ROLE_LABELS_NOT_ENTRA_OBO**, not authenticated enterprise approval.
The existing `presentation_runtime.list_approvals(...)` and
`presentation_runtime.approve_request(...)` operate on the shared live/local
persistent queue. Inspect their documented inputs privately before use; these
notebooks never automatically approve requests. Synthetic reviewer labels are
not authenticated humans. Function schemas may be visible in the Foundry portal,
but the playground does not execute this application's local function handlers.
Use the authorized application client loop, not a portal chat, to test execution.

Native scope is **read-only evidence checks and detailed portal/SDK guidance**.
There is no authorization here for publishing, native configuration, tenant
consent, RBAC, licensing or governance mutations. Telemetry export configuration
requires separate reviewed approval. Added live invocation cells require an
independent default-off authorization for already-configured Monitor/OTLP egress;
invocation consent alone does not permit export. No cell configures an exporter.
The six-scene initial verification budget is at most 18 model calls across the
entire initial verification, not 18 per notebook. Choose the cells you need and
subtract any prior calls; do not run all optional examples within that budget.

[Agent 365 SDK overview](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/agent-365-sdk?tabs=python)
states that supported platforms use native integration without adding the SDK.
The SDK extends an existing runtime with identity, governed Work IQ tooling, and
notifications; it does not build, host, or orchestrate agents.
[Microsoft OpenTelemetry Distro](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/microsoft-opentelemetry)
is the separate current observability integration. Do **not** add the deprecated
Agent 365 observability SDK. These references describe platform choices, not
packages installed or tenant capabilities validated by this notebook.

Official source repositories for further study:
[Microsoft Agent Framework](https://github.com/microsoft/agent-framework)
(agent orchestration), [Azure AI Projects SDK](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects)
(Foundry client contracts), [Agent 365 Python SDK](https://github.com/microsoft/Agent365-python)
(selected developer integrations), and
[Microsoft OpenTelemetry Python Distro](https://github.com/microsoft/opentelemetry-distro-python)
(current observability). Follow this repository's pinned environment rather than
installing arbitrary versions from sample pages. No upstream code is copied here.